In [ ]:
!nvidia-smi


Sat Sep  5 04:52:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -q -U transformers datasets accelerate peft trl bitsandbytes

In [ ]:
import torch
import transformers
import datasets
import peft
import trl

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("peft:", peft.__version__)
print("trl:", trl.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

torch: 2.11.0+cu128
transformers: 5.16.1
datasets: 5.0.1
peft: 0.20.0
trl: 1.12.0
CUDA available: True
GPU: Tesla T4


In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import json

with open("nord_train.jsonl", "r", encoding="utf-8") as f:
    first_row = json.loads(f.readline())

print("Dataset loaded successfully!")
print("Keys:", first_row.keys())
print("\nFirst example:")
print(json.dumps(first_row, indent=2, ensure_ascii=False))

Dataset loaded successfully!
Keys: dict_keys(['messages', 'type'])

First example:
{
  "messages": [
    {
      "role": "system",
      "content": "You are NORD, a concise industrial manual assistant. Use only supplied evidence. Never invent a code meaning."
    },
    {
      "role": "user",
      "content": "What does E116 mean on PRESS_GAMMA?"
    },
    {
      "role": "assistant",
      "content": "E116 on PRESS_GAMMA (HP-90) means Pressure sensor signal fault. Possible causes include sensor connection issue and sensor fault. Recommended checks are to inspect sensor connection and follow diagnostic procedure. Source: PRESS_GAMMA_synthetic_manual, page 163."
    }
  ],
  "type": "direct_lookup"
}


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

print("Model loaded successfully!")
print("Model:", model_name)
print("Device:", model.device)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Model loaded successfully!
Model: Qwen/Qwen3-0.6B
Device: cuda:0


In [ ]:
prompt = """You are NORD, a concise industrial manual assistant.
Use only supplied evidence. Never invent a code meaning.

Machine: PRESS_GAMMA
Error code: E116

What does this error mean?"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=False
)

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)

 Please explain in simple terms.

The error message is: "E116: Press Gamma is not in the system."

The error code is: E116.

The machine is: PRESS_GAMMA.

The machine is: PRESS_GAMMA.

The machine is: PRESS_GAMMA.

The machine is: PRESS_GAMMA.

The machine is: PRESS_GAMMA.

The machine is: PRESS_GAMMA.

The machine is: PRESS_GAMMA.

The machine


In [ ]:
import json

with open("nord_train.jsonl", "r", encoding="utf-8") as f:
    train_data = [json.loads(line) for line in f]

print("Training examples:", len(train_data))
print("First type:", train_data[0]["type"])
print("Last type:", train_data[-1]["type"])

Training examples: 770
First type: direct_lookup
Last type: ambiguity


In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="nord_train.jsonl",
    split="train"
)

print(dataset)
print("\nNumber of examples:", len(dataset))
print("\nColumns:", dataset.column_names)

Dataset({
    features: ['messages', 'type'],
    num_rows: 770
})

Number of examples: 770

Columns: ['messages', 'type']


In [ ]:
text = tokenizer.apply_chat_template(
    dataset[0]["messages"],
    tokenize=False,
    add_generation_prompt=False
)

print(text)

<|im_start|>system
You are NORD, a concise industrial manual assistant. Use only supplied evidence. Never invent a code meaning.<|im_end|>
<|im_start|>user
What does E116 mean on PRESS_GAMMA?<|im_end|>
<|im_start|>assistant
<think>

</think>

E116 on PRESS_GAMMA (HP-90) means Pressure sensor signal fault. Possible causes include sensor connection issue and sensor fault. Recommended checks are to inspect sensor connection and follow diagnostic procedure. Source: PRESS_GAMMA_synthetic_manual, page 163.<|im_end|>



In [ ]:
import torch

print(f"GPU allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"GPU reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")
print(f"GPU total:     {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

GPU allocated: 1.12 GB
GPU reserved:  1.57 GB
GPU total:     14.56 GB


In [ ]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ]
)

print("LoRA configuration created successfully!")
print(lora_config)

LoRA configuration created successfully!
LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.20.0', base_model_name_or_path=None, revision=None, inference_mode=False, r=8, target_modules={'q_proj', 'k_proj', 'v_proj', 'o_proj'}, exclude_modules=None, lora_alpha=16, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, velora_config=None, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, monteclora_config=None, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None, ensure_weight_tying=False)


In [ ]:
model.add_adapter(lora_config)

trainable_params = 0
all_params = 0

for param in model.parameters():
    all_params += param.numel()
    if param.requires_grad:
        trainable_params += param.numel()

print(f"Total parameters:     {all_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Trainable percentage: {100 * trainable_params / all_params:.2f}%")

Total parameters:     598,343,680
Trainable parameters: 2,293,760
Trainable percentage: 0.38%


In [ ]:
!pip install -q -U "torchao>=0.17.0"

In [ ]:
import torchao

print("torchao version:", torchao.__version__)

torchao version: 0.18.0


In [ ]:
model.add_adapter(lora_config)

trainable_params = 0
all_params = 0

for param in model.parameters():
    all_params += param.numel()
    if param.requires_grad:
        trainable_params += param.numel()

print(f"Total parameters:     {all_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Trainable percentage: {100 * trainable_params / all_params:.2f}%")

ValueError: Adapter with name default already exists. Please use a different name.

In [ ]:
from peft import get_peft_model

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 2,293,760 || all params: 598,343,680 || trainable%: 0.3834


/usr/local/lib/python3.13/dist-packages/peft/mapping_func.py:78: UserWarning: The PEFT config's `base_model_name_or_path` was renamed from 'Qwen/Qwen3-0.6B' to 'None'. Please ensure that the correct base model is loaded when loading this checkpoint.
  warnings.warn(


In [ ]:
from trl import SFTConfig

training_args = SFTConfig(
    output_dir="./nord_qwen",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    fp16=True,
    report_to="none",
)

print("Training configuration created successfully!")
print(training_args)

Training configuration created successfully!
SFTConfig(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
activation_offloading=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
assistant_only_loss=False,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
chat_template_path=None,
completion_only_loss=None,
data_seed=None,
dataloader_drop_last=False,
dataloader_in_order=True,
dataloader_multiprocessing_context=None,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
dataset_kwargs=None,
dataset_num_proc=None,
dataset_text_field=text,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
    peft_config=None,
)

print("Trainer created successfully!")

Tokenizing train dataset:   0%|          | 0/770 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/770 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/770 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/770 [00:00<?, ? examples/s]

Trainer created successfully!


In [ ]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


AttributeError: 'CausalLMOutputWithPast' object has no attribute 'last_hidden_state'

In [ ]:
prompt = """You are NORD, a concise industrial manual assistant.
Use only supplied evidence. Never invent a code meaning.

Machine: PRESS_GAMMA
Error code: E116

What does this error mean?"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=False
)

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
[transformers] Caching is incompatible with gradient checkpointing in Qwen3DecoderLayer. Setting `past_key_values=None`.


 Please Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Inst

In [ ]:
messages = [
    {
        "role": "system",
        "content": "You are NORD, a concise industrial manual assistant. Use only supplied evidence. Never invent a code meaning."
    },
    {
        "role": "user",
        "content": "What does E116 mean on PRESS_GAMMA?"
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id
    )

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)

<think>Question Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructi

In [ ]:
model.disable_adapter()

messages = [
    {
        "role": "system",
        "content": "You are NORD, a concise industrial manual assistant. Use only supplied evidence. Never invent a code meaning."
    },
    {
        "role": "user",
        "content": "What does E116 mean on PRESS_GAMMA?"
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id
    )

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)

<think>Question Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructi

In [ ]:
print("EOS token ID:", tokenizer.eos_token_id)
print("PAD token ID:", tokenizer.pad_token_id)
print("Model EOS:", model.config.eos_token_id)
print("Model PAD:", model.config.pad_token_id)
print("Generation EOS:", model.generation_config.eos_token_id)
print("Generation PAD:", model.generation_config.pad_token_id)

EOS token ID: 151645
PAD token ID: 151643
Model EOS: 151645
Model PAD: 151643
Generation EOS: [151645, 151643]
Generation PAD: 151643


In [ ]:
messages = [
    {
        "role": "system",
        "content": "You are NORD, a concise industrial manual assistant. Use only supplied evidence. Never invent a code meaning."
    },
    {
        "role": "user",
        "content": "What does E116 mean on PRESS_GAMMA?"
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

print(text)

<|im_start|>system
You are NORD, a concise industrial manual assistant. Use only supplied evidence. Never invent a code meaning.<|im_end|>
<|im_start|>user
What does E116 mean on PRESS_GAMMA?<|im_end|>
<|im_start|>assistant



In [ ]:
model.disable_adapter()

messages = [
    {
        "role": "system",
        "content": "You are NORD, a concise industrial manual assistant. Use only supplied evidence. Never invent a code meaning."
    },
    {
        "role": "user",
        "content": "What does E116 mean on PRESS_GAMMA?"
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)

Elicants Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Ins

In [ ]:
print("Transformers:", transformers.__version__)
print("Model class:", type(model).__name__)
print("Model name:", model.config._name_or_path)

print("\nQwen3 generation support:")
print("Has _supports_default_dynamic_cache:",
      hasattr(model, "_supports_default_dynamic_cache"))
print("Has generate:", hasattr(model, "generate"))

Transformers: 5.16.1
Model class: PeftModelForCausalLM
Model name: Qwen/Qwen3-0.6B

Qwen3 generation support:
Has _supports_default_dynamic_cache: True
Has generate: True


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

base_model_name = "Qwen/Qwen3-0.6B"

base_tokenizer = AutoTokenizer.from_pretrained(base_model_name)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Fresh base model loaded!")
print("Model class:", type(base_model).__name__)
print("Device:", base_model.device)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Fresh base model loaded!
Model class: Qwen3ForCausalLM
Device: cuda:0


In [ ]:
messages = [
    {
        "role": "system",
        "content": "You are a helpful assistant."
    },
    {
        "role": "user",
        "content": "What is 2 + 2? Answer briefly."
    }
]

text = base_tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

inputs = base_tokenizer(
    text,
    return_tensors="pt"
).to(base_model.device)

with torch.no_grad():
    outputs = base_model.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=False,
        pad_token_id=base_tokenizer.pad_token_id,
        eos_token_id=base_tokenizer.eos_token_id
    )

response = base_tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)

2 + 2 = 4.


In [ ]:
print("Model class:", type(model).__name__)

print("\nLoRA trainable parameters:")
model.print_trainable_parameters()

Model class: PeftModelForCausalLM

LoRA trainable parameters:
trainable params: 2,293,760 || all params: 598,343,680 || trainable%: 0.3834


In [ ]:
print("Processed dataset columns:")
print(trainer.train_dataset.column_names)

print("\nFirst processed example:")
example = trainer.train_dataset[0]

print("Input IDs:", example["input_ids"][:30])
print("Labels:   ", example["labels"][:30])

print("\nNumber of tokens:", len(example["input_ids"]))

Processed dataset columns:
['messages', 'type', 'input_ids', 'labels']

First processed example:
Input IDs: [151644, 8948, 198, 2610, 525, 451, 4276, 11, 264, 63594, 12785, 11376, 17847, 13, 5443, 1172, 17221, 5904, 13, 14695, 17023, 264, 2038, 7290, 13, 151645, 198, 151644, 872, 198]
Labels:    [151644, 8948, 198, 2610, 525, 451, 4276, 11, 264, 63594, 12785, 11376, 17847, 13, 5443, 1172, 17221, 5904, 13, 14695, 17023, 264, 2038, 7290, 13, 151645, 198, 151644, 872, 198]

Number of tokens: 113


In [ ]:
del trainer
del model

import gc
import torch

gc.collect()
torch.cuda.empty_cache()

print(f"GPU allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print("Old trainer/model cleared.")

GPU allocated: 2.62 GB
Old trainer/model cleared.


In [ ]:
import inspect
from trl import SFTConfig

print(inspect.signature(SFTConfig))

(output_dir: str | None = None, per_device_train_batch_size: int = 8, num_train_epochs: float = 3.0, max_steps: int = -1, learning_rate: float = 2e-05, lr_scheduler_type: transformers.trainer_utils.SchedulerType | str = 'linear', lr_scheduler_kwargs: dict | str | None = None, warmup_steps: float = 0, optim: transformers.training_args.OptimizerNames | str = 'adamw_torch_fused', optim_args: str | None = None, weight_decay: float = 0.0, adam_beta1: float = 0.9, adam_beta2: float = 0.999, adam_epsilon: float = 1e-08, optim_target_modules: None | str | list[str] = None, gradient_accumulation_steps: int = 1, average_tokens_across_devices: bool = True, max_grad_norm: float = 1.0, label_smoothing_factor: float = 0.0, bf16: bool | None = None, fp16: bool = False, bf16_full_eval: bool = False, fp16_full_eval: bool = False, tf32: bool | None = None, gradient_checkpointing: bool = True, gradient_checkpointing_kwargs: dict[str, typing.Any] | str | None = None, torch_compile: bool = False, torch_com

In [ ]:
from trl import SFTConfig

training_args = SFTConfig(
    output_dir="./nord_qwen_v2",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    fp16=True,
    report_to="none",
    assistant_only_loss=True,
)

print("Corrected training configuration created!")
print("assistant_only_loss:", training_args.assistant_only_loss)

Corrected training configuration created!
assistant_only_loss: True


In [ ]:
from peft import get_peft_model

model_v2 = get_peft_model(base_model, lora_config)

model_v2.print_trainable_parameters()

trainable params: 2,293,760 || all params: 598,343,680 || trainable%: 0.3834


In [ ]:
from trl import SFTTrainer

trainer_v2 = SFTTrainer(
    model=model_v2,
    args=training_args,
    train_dataset=dataset,
    processing_class=base_tokenizer,
)

print("Corrected trainer created successfully!")

Tokenizing train dataset:   0%|          | 0/770 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/770 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/770 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/770 [00:00<?, ? examples/s]

Corrected trainer created successfully!


In [ ]:
example = trainer_v2.train_dataset[0]

labels = example["labels"]

total_tokens = len(labels)
trained_tokens = sum(1 for x in labels if x != -100)
masked_tokens = sum(1 for x in labels if x == -100)

print("Total tokens:    ", total_tokens)
print("Masked tokens:   ", masked_tokens)
print("Trained tokens:  ", trained_tokens)
print("Masked %:        ", round(masked_tokens / total_tokens * 100, 2))

Total tokens:     113
Masked tokens:    48
Trained tokens:   65
Masked %:         42.48


In [ ]:
trainer_v2.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,2.737548
20,1.443628
30,0.799967
40,0.496783
50,0.366197
60,0.257816
70,0.201692
80,0.191889
90,0.168891
100,0.125065


TrainOutput(global_step=194, training_loss=0.39274172891968306, metrics={'train_runtime': 329.0628, 'train_samples_per_second': 4.68, 'train_steps_per_second': 0.59, 'total_flos': 470329300353024.0, 'train_loss': 0.39274172891968306, 'entropy': 0.09698548977478193, 'num_tokens': 167258.0, 'mean_token_accuracy': 0.9823323992582468, 'epoch': 2.0})

In [ ]:
messages = [
    {
        "role": "system",
        "content": "You are NORD, a concise industrial manual assistant. Use only supplied evidence. Never invent a code meaning."
    },
    {
        "role": "user",
        "content": "What does E116 mean on PRESS_GAMMA?"
    }
]

text = base_tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

inputs = base_tokenizer(
    text,
    return_tensors="pt"
).to(model_v2.device)

with torch.no_grad():
    outputs = model_v2.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
        pad_token_id=base_tokenizer.pad_token_id,
        eos_token_id=base_tokenizer.eos_token_id
    )

response = base_tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)

Eotional Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Ins

In [ ]:
model_v2.disable_adapter()

print("Adapter disabled.")
print("Model class:", type(model_v2).__name__)

Adapter disabled.
Model class: PeftModelForCausalLM


In [ ]:
messages = [
    {
        "role": "system",
        "content": "You are a helpful assistant."
    },
    {
        "role": "user",
        "content": "What is 2 + 2? Answer briefly."
    }
]

text = base_tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

inputs = base_tokenizer(
    text,
    return_tensors="pt"
).to(model_v2.device)

with torch.no_grad():
    outputs = model_v2.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=False,
        pad_token_id=base_tokenizer.pad_token_id,
        eos_token_id=base_tokenizer.eos_token_id
    )

response = base_tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)

2 Answer Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions


In [ ]:
import gc
import torch

# Remove all current model objects
for name in ["model_v2", "base_model"]:
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()

print(f"GPU allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print("Current model objects cleared.")

GPU allocated: 2.65 GB
Current model objects cleared.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

base_model_name = "Qwen/Qwen3-0.6B"

tokenizer_clean = AutoTokenizer.from_pretrained(base_model_name)

model_clean = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    dtype=torch.float16,
    device_map="auto"
)

print("Fresh model loaded!")
print("Class:", type(model_clean).__name__)
print("Device:", model_clean.device)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Fresh model loaded!
Class: Qwen3ForCausalLM
Device: cuda:0


In [ ]:
messages = [
    {
        "role": "system",
        "content": "You are a helpful assistant."
    },
    {
        "role": "user",
        "content": "What is 2 + 2? Answer briefly."
    }
]

text = tokenizer_clean.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

inputs = tokenizer_clean(
    text,
    return_tensors="pt"
).to(model_clean.device)

with torch.no_grad():
    outputs = model_clean.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=False,
        pad_token_id=tokenizer_clean.pad_token_id,
        eos_token_id=tokenizer_clean.eos_token_id
    )

response = tokenizer_clean.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)

2 + 2 = 4.


In [ ]:
def format_example(example):
    return {
        "text": tokenizer_clean.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=False
        )
    }

formatted_dataset = dataset.map(format_example)

print("Formatted dataset created!")
print(formatted_dataset.column_names)
print("\nFirst formatted example:\n")
print(formatted_dataset[0]["text"])

Map:   0%|          | 0/770 [00:00<?, ? examples/s]

Formatted dataset created!
['messages', 'type', 'text']

First formatted example:

<|im_start|>system
You are NORD, a concise industrial manual assistant. Use only supplied evidence. Never invent a code meaning.<|im_end|>
<|im_start|>user
What does E116 mean on PRESS_GAMMA?<|im_end|>
<|im_start|>assistant
<think>

</think>

E116 on PRESS_GAMMA (HP-90) means Pressure sensor signal fault. Possible causes include sensor connection issue and sensor fault. Recommended checks are to inspect sensor connection and follow diagnostic procedure. Source: PRESS_GAMMA_synthetic_manual, page 163.<|im_end|>



In [ ]:
think_count = sum(
    "<think>" in example["text"]
    for example in formatted_dataset
)

print("Examples containing <think>:", think_count)
print("Total examples:", len(formatted_dataset))

Examples containing <think>: 770
Total examples: 770


In [ ]:
def clean_thinking(example):
    example["text"] = example["text"].replace("<think>\n\n</think>\n\n", "")
    return example

clean_dataset = formatted_dataset.map(clean_thinking)

print("Cleaning complete!")

print("\nFirst cleaned example:\n")
print(clean_dataset[0]["text"])

print("\nRemaining <think> tags:",
      sum("<think>" in x["text"] for x in clean_dataset))

Map:   0%|          | 0/770 [00:00<?, ? examples/s]

Cleaning complete!

First cleaned example:

<|im_start|>system
You are NORD, a concise industrial manual assistant. Use only supplied evidence. Never invent a code meaning.<|im_end|>
<|im_start|>user
What does E116 mean on PRESS_GAMMA?<|im_end|>
<|im_start|>assistant
E116 on PRESS_GAMMA (HP-90) means Pressure sensor signal fault. Possible causes include sensor connection issue and sensor fault. Recommended checks are to inspect sensor connection and follow diagnostic procedure. Source: PRESS_GAMMA_synthetic_manual, page 163.<|im_end|>


Remaining <think> tags: 0


In [ ]:
empty_count = sum(
    len(example["text"].strip()) == 0
    for example in clean_dataset
)

print("Total examples:", len(clean_dataset))
print("Empty examples:", empty_count)
print("Examples with <think>:", sum(
    "<think>" in example["text"] for example in clean_dataset
))

Total examples: 770
Empty examples: 0
Examples with <think>: 0


In [ ]:
from peft import LoraConfig, get_peft_model

lora_config_v2 = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
)

model_nord = get_peft_model(model_clean, lora_config_v2)

model_nord.print_trainable_parameters()

trainable params: 2,293,760 || all params: 598,343,680 || trainable%: 0.3834


In [ ]:
from trl import SFTTrainer, SFTConfig

final_training_args = SFTConfig(
    output_dir="./nord_final",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    fp16=True,
    report_to="none",
    dataset_text_field="text",
    assistant_only_loss=True,
)

trainer_final = SFTTrainer(
    model=model_nord,
    args=final_training_args,
    train_dataset=clean_dataset,
    processing_class=tokenizer_clean,
)

print("Final NORD trainer created successfully!")

Tokenizing train dataset:   0%|          | 0/770 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/770 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/770 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/770 [00:00<?, ? examples/s]

Final NORD trainer created successfully!


In [ ]:
# Verify that loss is applied only to the assistant response

sample = trainer_final.train_dataset[0]

labels = sample["labels"]
input_ids = sample["input_ids"]

masked = sum(1 for x in labels if x == -100)
trained = sum(1 for x in labels if x != -100)

print("Total tokens :", len(input_ids))
print("Masked tokens:", masked)
print("Train tokens :", trained)

print("\nFirst 30 labels:")
print(labels[:30])

Total tokens : 113
Masked tokens: 48
Train tokens : 65

First 30 labels:
[-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100]


In [ ]:
train_result = trainer_final.train()

print("\nNORD training completed!")
print("Final training loss:", train_result.training_loss)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,2.727518
20,1.428285
30,0.795450
40,0.483281
50,0.360824
60,0.252043
70,0.193093
80,0.183444
90,0.169300
100,0.120706



NORD training completed!
Final training loss: 0.38814665507717233


In [ ]:

model_nord.save_pretrained("/content/nord_adapter")
tokenizer_clean.save_pretrained("/content/nord_adapter")

import shutil
shutil.make_archive("/content/nord_adapter", "zip", "/content/nord_adapter")

from google.colab import files
files.download("/content/nord_adapter.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
test_messages = [
    {
        "role": "system",
        "content": "You are NORD, a concise industrial manual assistant. Use only supplied evidence. Never invent a code meaning."
    },
    {
        "role": "user",
        "content": "What does E116 mean on PRESS_GAMMA?"
    }
]

prompt = tokenizer_clean.apply_chat_template(
    test_messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

inputs = tokenizer_clean(prompt, return_tensors="pt").to(model_nord.device)

with torch.no_grad():
    outputs = model_nord.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
        pad_token_id=tokenizer_clean.pad_token_id,
        eos_token_id=tokenizer_clean.eos_token_id
    )

new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
response = tokenizer_clean.decode(new_tokens, skip_special_tokens=True)

print(response)

Elicants Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Ins

In [ ]:
# Test whether the LoRA adapter is causing the repetition

from peft import PeftModel

# Disable the LoRA adapter
model_nord.disable_adapter()

test_messages = [
    {
        "role": "system",
        "content": "You are NORD, a concise industrial manual assistant."
    },
    {
        "role": "user",
        "content": "What does E116 mean on PRESS_GAMMA?"
    }
]

prompt = tokenizer_clean.apply_chat_template(
    test_messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

inputs = tokenizer_clean(prompt, return_tensors="pt").to(model_nord.device)

with torch.no_grad():
    outputs = model_nord.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=False,
        pad_token_id=tokenizer_clean.pad_token_id,
        eos_token_id=tokenizer_clean.eos_token_id
    )

new_tokens = outputs[0][inputs["input_ids"].shape[1]:]

print(tokenizer_clean.decode(new_tokens, skip_special_tokens=True))

Eotional Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen3-0.6B"

fresh_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

fresh_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
    device_map="auto"
)

messages = [
    {
        "role": "system",
        "content": "You are NORD, a concise industrial manual assistant."
    },
    {
        "role": "user",
        "content": "What does E116 mean on PRESS_GAMMA?"
    }
]

prompt = fresh_tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

inputs = fresh_tokenizer(prompt, return_tensors="pt").to(fresh_model.device)

with torch.no_grad():
    outputs = fresh_model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=False,
        pad_token_id=fresh_tokenizer.pad_token_id,
        eos_token_id=fresh_tokenizer.eos_token_id
    )

new_tokens = outputs[0][inputs["input_ids"].shape[1]:]

print(fresh_tokenizer.decode(new_tokens, skip_special_tokens=True))

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

E116 is a code used in the PRESS_GAMMA system to indicate a specific type of signal or data transmission. It typically represents a particular parameter or condition within the system. For more detailed information, it's best to consult the official


In [ ]:
model_nord.save_pretrained("/content/nord_adapter")
tokenizer_clean.save_pretrained("/content/nord_adapter")

import shutil
shutil.make_archive("/content/nord_adapter", "zip", "/content/nord_adapter")

from google.colab import files
files.download("/content/nord_adapter.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Inspect the magnitude of the trained LoRA weights

total = 0
count = 0
max_abs = 0

for name, param in model_nord.named_parameters():
    if "lora_" in name.lower():
        values = param.detach().float()

        total += values.abs().sum().item()
        count += values.numel()
        max_abs = max(max_abs, values.abs().max().item())

print("LoRA parameters:", count)
print("Mean absolute value:", total / count)
print("Maximum absolute value:", max_abs)

LoRA parameters: 2293760
Mean absolute value: 0.008564197734397436
Maximum absolute value: 0.044721875339746475


In [ ]:
# Compare next-token predictions with and without LoRA

model_nord.enable_adapter()

messages = [
    {
        "role": "system",
        "content": "You are NORD, a concise industrial manual assistant."
    },
    {
        "role": "user",
        "content": "What does E116 mean on PRESS_GAMMA?"
    }
]

prompt = tokenizer_clean.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

inputs = tokenizer_clean(prompt, return_tensors="pt").to(model_nord.device)

with torch.no_grad():
    logits = model_nord(**inputs).logits[:, -1, :]

top_values, top_ids = torch.topk(logits, 10, dim=-1)

print("Top 10 next tokens WITH LoRA:")
for value, token_id in zip(top_values[0], top_ids[0]):
    print(
        repr(tokenizer_clean.decode([token_id.item()])),
        "->",
        round(value.item(), 3)
    )

AttributeError: 'Qwen3ForCausalLM' object has no attribute 'enable_adapter'

In [ ]:
print("PEFT model type:", type(model_nord).__name__)

print("\nActive adapters:")
print(model_nord.active_adapters)

print("\nLoRA enabled:")
print(model_nord.peft_config)

print("\nTraining mode:", model_nord.training)


In [ ]:
messages = [
    {
        "role": "system",
        "content": "You are NORD, a concise industrial manual assistant."
    },
    {
        "role": "user",
        "content": "What does E116 mean on PRESS_GAMMA?"
    }
]

prompt = tokenizer_clean.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

inputs = tokenizer_clean(prompt, return_tensors="pt").to(model_nord.device)

# WITH LoRA
with torch.no_grad():
    logits_lora = model_nord(**inputs).logits[:, -1, :]

# WITHOUT LoRA
with model_nord.disable_adapter():
    with torch.no_grad():
        logits_base = model_nord(**inputs).logits[:, -1, :]

def show_top(logits, title):
    values, ids = torch.topk(logits, 10, dim=-1)
    print(f"\n{title}")
    for value, token_id in zip(values[0], ids[0]):
        token = tokenizer_clean.decode([token_id.item()])
        print(repr(token), "->", round(value.item(), 3))

show_top(logits_base, "BASE MODEL")
show_top(logits_lora, "WITH LORA")

In [ ]:
# Measure how strongly the LoRA adapter changes the model output

diff = (logits_lora.float() - logits_base.float()).abs()

print("Mean logit change :", diff.mean().item())
print("Max logit change  :", diff.max().item())

# Check the change for the top base-model token
base_top_id = torch.argmax(logits_base, dim=-1).item()
base_top_token = tokenizer_clean.decode([base_top_id])

print("\nBase top token:", repr(base_top_token))
print(
    "Base logit:",
    round(logits_base[0, base_top_id].item(), 3)
)
print(
    "LoRA logit:",
    round(logits_lora[0, base_top_id].item(), 3)
)

In [ ]:
# Test the model on an exact training example

example = clean_dataset[0]

print("TRAINING EXAMPLE:")
print(example["text"])

# Use only the system + user portion as the prompt
messages = dataset[0]["messages"][:2]

prompt = tokenizer_clean.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

inputs = tokenizer_clean(prompt, return_tensors="pt").to(model_nord.device)

model_nord.eval()

with torch.no_grad():
    outputs = model_nord.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
        pad_token_id=tokenizer_clean.pad_token_id,
        eos_token_id=tokenizer_clean.eos_token_id
    )

new_tokens = outputs[0][inputs["input_ids"].shape[1]:]

print("\nMODEL OUTPUT:")
print(tokenizer_clean.decode(new_tokens, skip_special_tokens=True))

In [ ]:
# Test on a held-out example the model never saw during training

eval_example = {
    "role": "user",
    "content": "What does E101 mean on CNC_ALPHA?"
}

messages = [
    {
        "role": "system",
        "content": "You are NORD, a concise industrial manual assistant. Use only supplied evidence. Never invent a code meaning."
    },
    eval_example
]

prompt = tokenizer_clean.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

inputs = tokenizer_clean(prompt, return_tensors="pt").to(model_nord.device)

model_nord.eval()

with torch.no_grad():
    outputs = model_nord.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
        pad_token_id=tokenizer_clean.pad_token_id,
        eos_token_id=tokenizer_clean.eos_token_id
    )

new_tokens = outputs[0][inputs["input_ids"].shape[1]:]

print(tokenizer_clean.decode(new_tokens, skip_special_tokens=True))

In [ ]:
from datasets import load_dataset

eval_dataset = load_dataset(
    "json",
    data_files="nord_eval.jsonl",
    split="train"
)

print("Evaluation examples:", len(eval_dataset))

for i in range(5):
    print("\n--- Example", i, "---")
    print(eval_dataset[i]["messages"])

In [ ]:
from google.colab import files

uploaded = files.upload()

print("Uploaded files:", list(uploaded.keys()))

In [ ]:
from datasets import load_dataset

eval_dataset = load_dataset(
    "json",
    data_files="nord_eval.jsonl",
    split="train"
)

print("Evaluation examples:", len(eval_dataset))

for i in range(5):
    print("\n--- Example", i, "---")
    print(eval_dataset[i]["messages"])

In [ ]:
def generate_nord(messages, max_new_tokens=100):
    prompt = tokenizer_clean.apply_chat_template(
        messages[:2],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer_clean(
        prompt,
        return_tensors="pt"
    ).to(model_nord.device)

    model_nord.eval()

    with torch.no_grad():
        outputs = model_nord.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer_clean.pad_token_id,
            eos_token_id=tokenizer_clean.eos_token_id
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer_clean.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()


for i, example in enumerate(eval_dataset):
    print(f"\n{'='*60}")
    print(f"TEST {i+1}")
    print("QUERY:", example["messages"][1]["content"])
    print("EXPECTED:", example["messages"][2]["content"])
    print("MODEL:", generate_nord(example["messages"]))

In [ ]:
import json

with open("/mnt/data/mendx_nord_dataset/nord_knowledge_base.json", "r") as f:
    kb = json.load(f)

print(json.dumps(kb, indent=2))

In [ ]:
import json

with open("nord_eval.jsonl", "r", encoding="utf-8") as f:
    eval_rows = [json.loads(line) for line in f]

for row in eval_rows[:4]:
    print(row["messages"][1]["content"])
    print("→", row["messages"][2]["content"])
    print()

In [ ]:
test_queries = [
    "What does E101 mean on CNC_ALPHA?",
    "What does E101 mean on PRESS_GAMMA?",
    "What does E101 mean on CNC_BETA?",
    "What does E101 mean on PRESS_DELTA?",
]

for query in test_queries:
    messages = [
        {
            "role": "system",
            "content": "You are NORD, a concise industrial manual assistant. Use only supplied evidence. Never invent a code meaning."
        },
        {"role": "user", "content": query}
    ]

    prompt = tokenizer_clean.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer_clean(prompt, return_tensors="pt").to(model_nord.device)

    with torch.no_grad():
        outputs = model_nord.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            pad_token_id=tokenizer_clean.pad_token_id,
            eos_token_id=tokenizer_clean.eos_token_id
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer_clean.decode(generated, skip_special_tokens=True)

    print("=" * 80)
    print(query)
    print("MODEL:", answer.strip())

In [ ]:
# Create a tiny synthetic knowledge base for NORD RAG

knowledge_base = [
    {
        "machine": "CNC_ALPHA",
        "model": "AX-200",
        "error_code": "E101",
        "meaning": "Spindle overload",
        "causes": "excessive cutting load and tool/workpiece issue",
        "checks": "inspect spindle load and verify tooling and workpiece setup",
        "source": "CNC_ALPHA_synthetic_manual",
        "page": 71
    },
    {
        "machine": "PRESS_GAMMA",
        "model": "HP-90",
        "error_code": "E101",
        "meaning": "Hydraulic pressure loss",
        "causes": "low hydraulic level and pump issue",
        "checks": "check hydraulic level and inspect pump and visible leaks",
        "source": "PRESS_GAMMA_synthetic_manual",
        "page": 71
    },
    {
        "machine": "CNC_BETA",
        "model": "BX-500",
        "error_code": "E101",
        "meaning": "Coolant pump fault",
        "causes": "pump failure and coolant flow restriction",
        "checks": "inspect coolant level, pump operation, and flow path",
        "source": "CNC_BETA_synthetic_manual",
        "page": 71
    },
    {
        "machine": "PRESS_DELTA",
        "model": "PD-40",
        "error_code": "E101",
        "meaning": "Pressure sensor signal fault",
        "causes": "sensor connection issue and sensor fault",
        "checks": "inspect sensor connection and follow diagnostic procedure",
        "source": "PRESS_DELTA_synthetic_manual",
        "page": 71
    }
]

import json

with open("nord_knowledge_base.json", "w", encoding="utf-8") as f:
    json.dump(knowledge_base, f, indent=2)

print("Knowledge base created:", len(knowledge_base), "entries")
print(json.dumps(knowledge_base[0], indent=2))

Knowledge base created: 4 entries
{
  "machine": "CNC_ALPHA",
  "model": "AX-200",
  "error_code": "E101",
  "meaning": "Spindle overload",
  "causes": "excessive cutting load and tool/workpiece issue",
  "checks": "inspect spindle load and verify tooling and workpiece setup",
  "source": "CNC_ALPHA_synthetic_manual",
  "page": 71
}


In [ ]:
   import json
   with open("nord_knowledge_base.json", "w", encoding="utf-8") as f:
       json.dump(knowledge_base, f, indent=2)

   from google.colab import files
   files.download("nord_knowledge_base.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import json

with open("nord_knowledge_base.json", "r", encoding="utf-8") as f:
    kb = json.load(f)

def retrieve(machine, error_code):
    for item in kb:
        if item["machine"] == machine and item["error_code"] == error_code:
            return item
    return None

for machine in ["CNC_ALPHA", "PRESS_GAMMA", "CNC_BETA", "PRESS_DELTA"]:
    result = retrieve(machine, "E101")
    print(f"{machine} + E101 → {result['meaning']}")

print("\nUnknown machine test:")
print(retrieve("UNKNOWN_MACHINE", "E101"))

In [ ]:
def rag_answer(machine, error_code):
    # 1. Retrieve evidence
    evidence = retrieve(machine, error_code)

    if evidence is None:
        return "I cannot find this error code for the specified machine in the available manual evidence."

    # 2. Build grounded prompt
    evidence_text = f"""
Machine: {evidence['machine']}
Model: {evidence['model']}
Error Code: {evidence['error_code']}
Meaning: {evidence['meaning']}
Possible Causes: {evidence['causes']}
Recommended Checks: {evidence['checks']}
Source: {evidence['source']}, page {evidence['page']}
"""

    messages = [
        {
            "role": "system",
            "content": """You are NORD, a concise industrial manual assistant.
Answer ONLY using the supplied manual evidence.
Do not change the machine, error code, meaning, causes, checks, or source.
If the evidence does not contain the answer, say you cannot find it."""
        },
        {
            "role": "user",
            "content": f"""What does {error_code} mean on {machine}?

SUPPLIED MANUAL EVIDENCE:
{evidence_text}"""
        }
    ]

    prompt = tokenizer_clean.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer_clean(prompt, return_tensors="pt").to(model_nord.device)

    with torch.no_grad():
        outputs = model_nord.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,
            pad_token_id=tokenizer_clean.pad_token_id,
            eos_token_id=tokenizer_clean.eos_token_id
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer_clean.decode(
        generated,
        skip_special_tokens=True
    ).strip()


# Test all four machine-specific cases
for machine in ["CNC_ALPHA", "PRESS_GAMMA", "CNC_BETA", "PRESS_DELTA"]:
    print("=" * 80)
    print(f"QUERY: What does E101 mean on {machine}?")
    print("ANSWER:", rag_answer(machine, "E101"))

In [ ]:
print(rag_answer("UNKNOWN_MACHINE", "E101"))

In [ ]:
print(rag_answer("", "E101"))

In [ ]:
def retrieve_by_error(error_code):
    return [
        item for item in kb
        if item["error_code"] == error_code
    ]

results = retrieve_by_error("E101")

print("Matches found:", len(results))

for item in results:
    print(
        f"{item['machine']} ({item['model']}) → "
        f"{item['meaning']}"
    )

In [ ]:
def retrieve_nord(machine, error_code):
    # Machine specified → exact lookup
    if machine:
        for item in kb:
            if (
                item["machine"].lower() == machine.lower()
                and item["error_code"].lower() == error_code.lower()
            ):
                return {
                    "status": "found",
                    "evidence": item
                }

        return {
            "status": "not_found",
            "evidence": None
        }

    # Machine not specified → find all matching error codes
    matches = [
        item for item in kb
        if item["error_code"].lower() == error_code.lower()
    ]

    if len(matches) == 0:
        return {
            "status": "not_found",
            "evidence": None
        }

    if len(matches) > 1:
        return {
            "status": "ambiguous",
            "evidence": matches
        }

    return {
        "status": "found",
        "evidence": matches[0]
    }


result = retrieve_nord("", "E101")

print("Status:", result["status"])

if result["status"] == "ambiguous":
    print("\nNORD should ask the user to specify one of:")
    for item in result["evidence"]:
        print(f"- {item['machine']} ({item['model']})")

In [ ]:
def nord_route(machine, error_code):
    result = retrieve_nord(machine, error_code)

    if result["status"] == "not_found":
        return "I cannot find this error code in the available manual evidence."

    if result["status"] == "ambiguous":
        machines = [
            f"{item['machine']} ({item['model']})"
            for item in result["evidence"]
        ]

        return (
            f"{error_code} has different meanings across the available machines. "
            f"Please specify the machine or model. Available matches: "
            + ", ".join(machines) + "."
        )

    evidence = result["evidence"]

    return (
        f"{error_code} on {evidence['machine']} ({evidence['model']}) "
        f"means {evidence['meaning']}. "
        f"Possible causes include {evidence['causes']}. "
        f"Recommended checks are to {evidence['checks']}. "
        f"Source: {evidence['source']}, page {evidence['page']}."
    )


print(nord_route("", "E101"))

In [ ]:
test_cases = [
    ("CNC_ALPHA", "E101"),
    ("", "E101"),
    ("UNKNOWN_MACHINE", "E101")
]

for machine, error_code in test_cases:
    print("=" * 80)
    print(f"Machine: {machine or '[NOT PROVIDED]'}")
    print(f"Error:   {error_code}")
    print("NORD:", nord_route(machine, error_code))

In [ ]:
def nord(query, machine=None, error_code=None):
    """
    NORD:
    Query → identify machine/error → retrieve evidence →
    handle ambiguity/not-found → grounded response
    """

    # If machine and error code are already extracted
    if error_code is None:
        import re
        match = re.search(r"\bE\d{3}\b", query.upper())
        error_code = match.group(0) if match else None

    if error_code is None:
        return "I could not identify an error code from the query."

    result = retrieve_nord(machine or "", error_code)

    # No matching evidence
    if result["status"] == "not_found":
        return "I cannot find this error code in the available manual evidence."

    # Multiple possible machines
    if result["status"] == "ambiguous":
        machines = [
            f"{item['machine']} ({item['model']})"
            for item in result["evidence"]
        ]

        return (
            f"{error_code} has different meanings across the available machines. "
            f"Please specify the machine or model. "
            f"Available matches: {', '.join(machines)}."
        )

    # Exact evidence found
    evidence = result["evidence"]

    return (
        f"{error_code} on {evidence['machine']} ({evidence['model']}) "
        f"means {evidence['meaning']}. "
        f"Possible causes include {evidence['causes']}. "
        f"Recommended checks are to {evidence['checks']}. "
        f"Source: {evidence['source']}, page {evidence['page']}."
    )


# Final quick test
print(nord("What does E101 mean on CNC_ALPHA?", machine="CNC_ALPHA"))

In [ ]:
import re

def extract_error_code(query):
    match = re.search(r"\bE\d{3}\b", query.upper())
    return match.group(0) if match else None


def extract_machine(query):
    query_upper = query.upper()

    for item in kb:
        if item["machine"].upper() in query_upper:
            return item["machine"]

    return None


def parse_nord_query(query):
    error_code = extract_error_code(query)
    machine = extract_machine(query)

    return {
        "query": query,
        "machine": machine,
        "error_code": error_code
    }


queries = [
    "What does E101 mean on CNC_ALPHA?",
    "Can you explain error E101 on PRESS_GAMMA?",
    "I am getting E101 on CNC_BETA",
    "PRESS_DELTA is showing error E101",
    "What does E101 mean?"
]

for q in queries:
    print(parse_nord_query(q))

In [ ]:
def nord_query(query):
    parsed = parse_nord_query(query)

    return nord(
        query=parsed["query"],
        machine=parsed["machine"],
        error_code=parsed["error_code"]
    )


test_queries = [
    "What does E101 mean on CNC_ALPHA?",
    "Can you explain error E101 on PRESS_GAMMA?",
    "I am getting E101 on CNC_BETA",
    "PRESS_DELTA is showing error E101",
    "What does E101 mean?",
    "What does E999 mean on CNC_ALPHA?"
]

for query in test_queries:
    print("=" * 80)
    print("QUERY:", query)
    print("NORD:", nord_query(query))

In [ ]:
!pip -q install sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully")
print("Embedding dimension:", embedding_model.get_sentence_embedding_dimension())

In [ ]:
# Generate embeddings for the NORD knowledge base

kb_texts = [
    f"{item['machine']} {item['model']} {item['error_code']} "
    f"{item['meaning']} {item['causes']} {item['checks']}"
    for item in kb
]

kb_embeddings = embedding_model.encode(
    kb_texts,
    convert_to_tensor=True,
    normalize_embeddings=True
)

print("Knowledge-base entries:", len(kb_texts))
print("Embedding shape:", kb_embeddings.shape)
print("First embedding (first 5 values):", kb_embeddings[0][:5])

In [ ]:
import torch

def semantic_retrieve(query, top_k=4):
    query_embedding = embedding_model.encode(
        query,
        convert_to_tensor=True,
        normalize_embeddings=True
    )

    scores = torch.matmul(kb_embeddings, query_embedding)

    top_scores, top_indices = torch.topk(
        scores,
        k=min(top_k, len(kb))
    )

    results = []

    for score, index in zip(top_scores, top_indices):
        item = kb[index.item()]
        results.append({
            "machine": item["machine"],
            "model": item["model"],
            "error_code": item["error_code"],
            "meaning": item["meaning"],
            "score": float(score)
        })

    return results


query = "What does E101 mean on CNC_ALPHA?"

results = semantic_retrieve(query)

for result in results:
    print(
        f"{result['machine']} ({result['model']}) → "
        f"{result['meaning']} | similarity: {result['score']:.4f}"
    )

In [ ]:
def hybrid_retrieve(query, top_k=4):
    query_embedding = embedding_model.encode(
        query,
        convert_to_tensor=True,
        normalize_embeddings=True
    )

    semantic_scores = torch.matmul(kb_embeddings, query_embedding)

    machine = extract_machine(query)
    error_code = extract_error_code(query)

    results = []

    for i, item in enumerate(kb):
        score = float(semantic_scores[i])

        # Strong bonus for exact machine match
        if machine and item["machine"].lower() == machine.lower():
            score += 1.0

        # Strong bonus for exact error-code match
        if error_code and item["error_code"].lower() == error_code.lower():
            score += 0.5

        results.append({
            "machine": item["machine"],
            "model": item["model"],
            "error_code": item["error_code"],
            "meaning": item["meaning"],
            "score": score
        })

    results.sort(key=lambda x: x["score"], reverse=True)

    return results[:top_k]


query = "What does E101 mean on CNC_ALPHA?"

results = hybrid_retrieve(query)

for result in results:
    print(
        f"{result['machine']} ({result['model']}) → "
        f"{result['meaning']} | hybrid score: {result['score']:.4f}"
    )

In [ ]:
test_queries = [
    "What does E101 mean on CNC_ALPHA?",
    "What does E101 mean on PRESS_GAMMA?",
    "What does E101 mean on CNC_BETA?",
    "What does E101 mean on PRESS_DELTA?"
]

for query in test_queries:
    print("=" * 80)
    print("QUERY:", query)

    results = hybrid_retrieve(query, top_k=1)
    best = results[0]

    print(
        f"TOP RESULT: {best['machine']} ({best['model']}) "
        f"→ {best['meaning']} | score: {best['score']:.4f}"
    )

In [ ]:
query = "What does E101 mean?"

results = hybrid_retrieve(query, top_k=4)

print("Query:", query)
print()

for result in results:
    print(
        f"{result['machine']} ({result['model']}) → "
        f"{result['meaning']} | score: {result['score']:.4f}"
    )

In [ ]:
def nord_hybrid_query(query):
    # 1. Extract identifiers
    parsed = parse_nord_query(query)

    machine = parsed["machine"]
    error_code = parsed["error_code"]

    if error_code is None:
        return {
            "status": "missing_error_code",
            "answer": "I could not identify an error code from the query."
        }

    # 2. If machine is missing, check exact code matches FIRST
    if machine is None:
        matches = retrieve_by_error(error_code)

        if len(matches) > 1:
            machines = [
                f"{item['machine']} ({item['model']})"
                for item in matches
            ]

            return {
                "status": "ambiguous",
                "answer": (
                    f"{error_code} has different meanings across the available "
                    f"machines. Please specify the machine or model. "
                    f"Available matches: {', '.join(machines)}."
                )
            }

    # 3. Machine is known → use hybrid retrieval
    results = hybrid_retrieve(query, top_k=1)

    if not results:
        return {
            "status": "not_found",
            "answer": "I cannot find this error code in the available manual evidence."
        }

    best = results[0]

    # 4. Verify that the retrieved result matches the requested identifiers
    if machine and best["machine"].lower() != machine.lower():
        return {
            "status": "not_found",
            "answer": "I cannot find this error code for the specified machine."
        }

    if error_code and best["error_code"].lower() != error_code.lower():
        return {
            "status": "not_found",
            "answer": "I cannot find this error code in the available manual evidence."
        }

    return {
        "status": "found",
        "evidence": best,
        "answer": (
            f"{error_code} on {best['machine']} ({best['model']}) "
            f"→ {best['meaning']}"
        )
    }


print(nord_hybrid_query("What does E101 mean on CNC_ALPHA?"))
print()
print(nord_hybrid_query("What does E101 mean?"))

In [ ]:
def nord_hybrid_answer(query):
    result = nord_hybrid_query(query)

    if result["status"] != "found":
        return result["answer"]

    evidence = result["evidence"]

    return (
        f"{evidence['error_code']} on {evidence['machine']} "
        f"({evidence['model']}) means {evidence['meaning']}. "
        f"Possible causes include {evidence['causes']}. "
        f"Recommended checks are to {evidence['checks']}. "
        f"Source: {kb[[x['machine'] for x in kb].index(evidence['machine'])]['source']}, "
        f"page {kb[[x['machine'] for x in kb].index(evidence['machine'])]['page']}."
    )


print(nord_hybrid_answer(
    "What does E101 mean on CNC_ALPHA?"
))

In [ ]:
def nord_hybrid_answer(query):
    result = nord_hybrid_query(query)

    if result["status"] != "found":
        return result["answer"]

    # Get the complete original KB entry
    machine = result["evidence"]["machine"]
    error_code = result["evidence"]["error_code"]

    full_evidence = next(
        (
            item for item in kb
            if item["machine"] == machine
            and item["error_code"] == error_code
        ),
        None
    )

    if full_evidence is None:
        return "I cannot find the complete manual evidence."

    return (
        f"{full_evidence['error_code']} on "
        f"{full_evidence['machine']} ({full_evidence['model']}) "
        f"means {full_evidence['meaning']}. "
        f"Possible causes include {full_evidence['causes']}. "
        f"Recommended checks are to {full_evidence['checks']}. "
        f"Source: {full_evidence['source']}, "
        f"page {full_evidence['page']}."
    )


print(nord_hybrid_answer(
    "What does E101 mean on CNC_ALPHA?"
))

In [ ]:
from google.colab import files

uploaded = files.upload()

print("Uploaded files:")
for filename in uploaded:
    print("-", filename)

In [ ]:
from pypdf import PdfReader

pdf_path = "test_manual_delta.pdf"

reader = PdfReader(pdf_path)

manual_pages = []

for page_number, page in enumerate(reader.pages, start=1):
    text = page.extract_text() or ""

    manual_pages.append({
        "page": page_number,
        "text": text.strip()
    })

print("Pages extracted:", len(manual_pages))

for page in manual_pages[:3]:
    print("=" * 70)
    print("PAGE:", page["page"])
    print(page["text"][:1000])

In [ ]:
!pip -q install pypdf

In [ ]:
from pypdf import PdfReader

pdf_path = "test_manual_delta.pdf"

reader = PdfReader(pdf_path)

manual_pages = []

for page_number, page in enumerate(reader.pages, start=1):
    text = page.extract_text() or ""

    manual_pages.append({
        "page": page_number,
        "text": text.strip()
    })

print("Pages extracted:", len(manual_pages))

for page in manual_pages[:3]:
    print("=" * 70)
    print("PAGE:", page["page"])
    print(manual_pages[page["page"] - 1]["text"][:1000])

In [ ]:
import re

chunks = []

for page in manual_pages:
    page_num = page["page"]
    text = page["text"]

    # Split on blank lines
    sections = re.split(r"\n\s*\n", text)

    for section in sections:
        section = section.strip()

        if len(section) < 80:
            continue

        chunks.append({
            "page": page_num,
            "text": section
        })

print("Total chunks:", len(chunks))

for i, chunk in enumerate(chunks[:10]):
    print("=" * 70)
    print("CHUNK:", i)
    print("PAGE:", chunk["page"])
    print(chunk["text"][:500])

In [ ]:
for i, chunk in enumerate(chunks):
    if "F0" in chunk["text"] or "Fault Codes" in chunk["text"]:
        print("=" * 70)
        print("CHUNK:", i)
        print("PAGE:", chunk["page"])
        print(chunk["text"])

In [ ]:
import re

fault_records = []
current = None

for page in manual_pages:
    page_num = page["page"]
    text = page["text"]

    lines = text.splitlines()

    for line in lines:
        # Detect a new fault code such as F001 — ...
        match = re.match(r"^\s*(F\d{3})\s*[—-]\s*(.+)", line)

        if match:
            # Save previous fault
            if current:
                current["text"] = current["text"].strip()
                current["page_end"] = page_num - 1
                fault_records.append(current)

            current = {
                "error_code": match.group(1),
                "title": match.group(2).strip(),
                "page_start": page_num,
                "page_end": page_num,
                "text": line.strip()
            }

        elif current:
            # Ignore the Fault Codes heading
            if line.strip() == "5. Fault Codes":
                continue

            current["text"] += "\n" + line

# Save final fault
if current:
    current["text"] = current["text"].strip()
    current["page_end"] = manual_pages[-1]["page"]
    fault_records.append(current)

print("Total fault records:", len(fault_records))

for fault in fault_records:
    print("=" * 70)
    print(fault["error_code"], "|", fault["title"])
    print("Pages:", fault["page_start"], "-", fault["page_end"])
    print(fault["text"][:500])

In [ ]:
# Rebuild fault records using the actual fault-code boundaries
# instead of relying on page numbers.

full_manual_text = "\n".join(
    f"\n--- PAGE {p['page']} ---\n{p['text']}"
    for p in manual_pages
)

matches = list(re.finditer(
    r"(?m)^\s*(F\d{3})\s*[—-]\s*(.+?)\s*$",
    full_manual_text
))

fault_records = []

for i, match in enumerate(matches):
    start = match.start()
    end = matches[i + 1].start() if i + 1 < len(matches) else len(full_manual_text)

    record_text = full_manual_text[start:end].strip()

    fault_records.append({
        "error_code": match.group(1),
        "title": match.group(2).strip(),
        "text": record_text
    })

print("Total fault records:", len(fault_records))

for fault in fault_records[:5]:
    print("=" * 70)
    print(fault["error_code"], "|", fault["title"])
    print(fault["text"][:1000])

In [ ]:
# Add machine/manual metadata to every fault record

for fault in fault_records:
    fault["machine"] = "Delta DX-200"
    fault["model"] = "DX-200"
    fault["manufacturer"] = "DeltaWorks Industries"
    fault["source"] = "test_manual_delta.pdf"

# Check one complete record
import json

print(json.dumps(fault_records[0], indent=2))

In [ ]:
# Create clean searchable documents for the RAG system

rag_documents = []

for fault in fault_records:
    rag_text = f"""
Machine: {fault["machine"]}
Model: {fault["model"]}
Manufacturer: {fault["manufacturer"]}
Error Code: {fault["error_code"]}
Fault: {fault["title"]}

{fault["text"]}
""".strip()

    rag_documents.append({
        "id": f'{fault["model"]}_{fault["error_code"]}',
        "text": rag_text,
        "metadata": {
            "machine": fault["machine"],
            "model": fault["model"],
            "manufacturer": fault["manufacturer"],
            "error_code": fault["error_code"],
            "title": fault["title"],
            "source": fault["source"]
        }
    })

print("RAG documents:", len(rag_documents))

print("\n" + "=" * 70)
print(rag_documents[0]["text"])

In [ ]:
# Create embeddings for all RAG documents

from sentence_transformers import SentenceTransformer
import torch

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device="cuda"
)

texts = [doc["text"] for doc in rag_documents]

rag_embeddings = embedding_model.encode(
    texts,
    convert_to_tensor=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Embedding shape:", rag_embeddings.shape)
print("Device:", rag_embeddings.device)
print("Embedding dimension:", rag_embeddings.shape[1])

In [ ]:
def semantic_retrieve(query, top_k=3):
    # Convert query into an embedding
    query_embedding = embedding_model.encode(
        query,
        convert_to_tensor=True,
        normalize_embeddings=True
    )

    # Cosine similarity because embeddings are normalized
    scores = torch.matmul(rag_embeddings, query_embedding)

    # Get highest scoring documents
    top_scores, top_indices = torch.topk(scores, k=top_k)

    results = []

    for score, idx in zip(top_scores, top_indices):
        doc = rag_documents[idx.item()]

        results.append({
            "score": round(float(score), 4),
            "error_code": doc["metadata"]["error_code"],
            "fault": doc["metadata"]["title"],
            "text": doc["text"]
        })

    return results


# Test with a natural-language troubleshooting query
query = "The machine shows F001 and the backgauge stops. What should I check?"

results = semantic_retrieve(query, top_k=3)

for i, result in enumerate(results, 1):
    print("=" * 70)
    print("RANK:", i)
    print("SCORE:", result["score"])
    print("ERROR:", result["error_code"])
    print("FAULT:", result["fault"])

In [ ]:
def hybrid_retrieve(query, top_k=3):
    # Semantic similarity
    query_embedding = embedding_model.encode(
        query,
        convert_to_tensor=True,
        normalize_embeddings=True
    )

    semantic_scores = torch.matmul(
        rag_embeddings,
        query_embedding
    )

    # Detect an explicit fault code
    code_match = re.search(r"\bF\d{3}\b", query.upper())
    query_code = code_match.group(0) if code_match else None

    results = []

    for i, doc in enumerate(rag_documents):
        semantic_score = float(semantic_scores[i])

        # Exact fault-code boost
        code_match_score = 1.0 if (
            query_code and
            doc["metadata"]["error_code"].upper() == query_code
        ) else 0.0

        final_score = semantic_score + code_match_score

        results.append({
            "score": round(final_score, 4),
            "semantic_score": round(semantic_score, 4),
            "code_match": bool(code_match_score),
            "error_code": doc["metadata"]["error_code"],
            "fault": doc["metadata"]["title"],
            "text": doc["text"]
        })

    results.sort(key=lambda x: x["score"], reverse=True)

    return results[:top_k]


# Test the same query
query = "The machine shows F001 and the backgauge stops. What should I check?"

results = hybrid_retrieve(query, top_k=3)

for i, result in enumerate(results, 1):
    print("=" * 70)
    print("RANK:", i)
    print("FINAL SCORE:", result["score"])
    print("SEMANTIC SCORE:", result["semantic_score"])
    print("EXACT CODE MATCH:", result["code_match"])
    print("ERROR:", result["error_code"])
    print("FAULT:", result["fault"])

In [ ]:
test_queries = [
    "The backgauge suddenly stops during deceleration and the machine shows a voltage fault",
    "The DC link voltage is too high on the machine",
    "The braking resistor may have failed and the backgauge is stopping",
    "The hydraulic motor is getting very hot",
    "The rear guard door is open and the machine will not start"
]

for query in test_queries:
    print("\n" + "=" * 80)
    print("QUERY:", query)

    results = hybrid_retrieve(query, top_k=1)

    result = results[0]

    print("TOP RESULT:", result["error_code"])
    print("FAULT:", result["fault"])
    print("SCORE:", result["score"])
    print("EXACT CODE MATCH:", result["code_match"])

In [ ]:
query = "The machine shows F001 and the backgauge stops. What should I check?"

result = hybrid_retrieve(query, top_k=1)[0]

print("=" * 80)
print("RETRIEVED FAULT")
print("=" * 80)
print("Error Code :", result["error_code"])
print("Fault      :", result["fault"])
print("Score      :", result["score"])
print()
print("FULL EVIDENCE")
print("=" * 80)
print(result["text"])

In [ ]:
def generate_nord_answer(query):
    # Retrieve the best manual evidence
    result = hybrid_retrieve(query, top_k=1)[0]

    prompt = f"""You are NORD, an industrial machine troubleshooting assistant.

Answer the user's question using ONLY the supplied manual evidence.

Rules:
- Do not invent information.
- Do not use outside knowledge.
- Give a concise troubleshooting answer.
- Mention the fault code and fault name.
- Give the relevant corrective steps.
- If the evidence does not answer the question, say:
  "I cannot find this information in the available manual."
- Always cite the source document and page information when available.

USER QUERY:
{query}

MANUAL EVIDENCE:
{result["text"]}

ANSWER:
"""

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=180,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = output[0][inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(
        generated,
        skip_special_tokens=True
    )

    return answer.strip()


# Test with the real manual
query = "The machine shows F001 and the backgauge stops. What should I check?"

answer = generate_nord_answer(query)

print("=" * 80)
print("USER QUERY")
print(query)
print("=" * 80)
print("NORD ANSWER")
print(answer)

In [ ]:
def generate_nord_answer(query):
    result = hybrid_retrieve(query, top_k=1)[0]

    prompt = f"""You are NORD, an industrial machine troubleshooting assistant.

Use ONLY the manual evidence below.

Return ONLY the final answer to the user.
Do NOT show reasoning.
Do NOT write <think>.
Do NOT explain how you reached the answer.

Keep the answer concise.

Required format:
Fault: <fault code and name>

Checks:
1. <step>
2. <step>
3. <step>

Source: <manual name and page if available>

USER QUERY:
{query}

MANUAL EVIDENCE:
{result["text"]}

FINAL ANSWER:
"""

    messages = [{"role": "user", "content": prompt}]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = output[0][inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

    # Remove any accidental thinking block
    if "<think>" in answer:
        answer = answer.split("</think>")[-1].strip()

    return answer


query = "The machine shows F001 and the backgauge stops. What should I check?"

answer = generate_nord_answer(query)

print("=" * 80)
print("USER QUERY")
print(query)
print("=" * 80)
print("NORD ANSWER")
print(answer)

In [ ]:
================================================================================
USER QUERY
The machine shows F001 and the backgauge stops. What should I check?
================================================================================
NORD ANSWER
<think>
Okay, let's see. The user mentioned that the machine shows F001 and the backgauge stops. The error code is DC Link Voltage Too High. From the manual, the probable cause is that the braking resistor is open-circuited or the deceleration ramp is too aggressive.

First, I need to check the braking resistor. The manual says to measure the resistance at the terminals R+ and RB. If it's too high, replace it. The resistance should be 47 Ω ±10%. If that's okay, then adjust the dec


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_NAME = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16
).to("cuda")

model.eval()

print("Model loaded successfully")
print("Device:", model.device)

In [ ]:
query = "The machine shows F001 and the backgauge stops. What should I check?"

result = hybrid_retrieve(query, top_k=1)[0]

prompt = f"""You are NORD, an industrial machine troubleshooting assistant.

Use ONLY the manual evidence below.

Give ONLY the final answer.
Do not show reasoning.
Do not use think tags.

USER QUERY:
{query}

MANUAL EVIDENCE:
{result["text"]}

FINAL ANSWER:
"""

messages = [{"role": "user", "content": prompt}]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=180,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

generated = output[0][inputs["input_ids"].shape[1]:]

answer = tokenizer.decode(
    generated,
    skip_special_tokens=True
).strip()

print("=" * 80)
print("NORD ANSWER")
print("=" * 80)
print(answer)

In [ ]:
prompt = f"""You are NORD, an industrial machine troubleshooting assistant.

Use ONLY the supplied manual evidence.

Answer the user's question with the specific corrective steps from the manual.
Do not invent information.
Do not show reasoning.
Do not use <think> tags.

Include:
- Fault code and fault name
- Corrective checks in the same order as the manual
- Important safety instruction if the manual specifies one
- Source/page information when available

Keep the answer concise.

USER QUERY:
{query}

MANUAL EVIDENCE:
{result["text"]}

FINAL ANSWER:
"""

messages = [{"role": "user", "content": prompt}]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=220,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

generated = output[0][inputs["input_ids"].shape[1]:]

answer = tokenizer.decode(
    generated,
    skip_special_tokens=True
).strip()

print("=" * 80)
print("NORD ANSWER")
print("=" * 80)
print(answer)

In [ ]:
# ============================================================
# NORD — RAG GROUNDED ANSWER GENERATION
# ============================================================

query = "The machine shows F001 and the backgauge stops. What should I check?"

# ------------------------------------------------------------
# 1. Retrieve the best manual evidence
# ------------------------------------------------------------

result = hybrid_retrieve(query, top_k=1)[0]

print("=" * 80)
print("RETRIEVED EVIDENCE")
print("=" * 80)
print("Error Code :", result["error_code"])
print("Fault      :", result["fault"])
print("Score      :", result["score"])
print()


# ------------------------------------------------------------
# 2. Build grounded prompt
# ------------------------------------------------------------

prompt = f"""You are NORD, an industrial machine troubleshooting assistant.

Use ONLY the supplied manual evidence.

Answer the user's question using the corrective steps given in the manual.

Rules:
- Do not invent information.
- Do not use outside knowledge.
- Do not show reasoning or analysis.
- Do not use <think> tags.
- Mention the fault code and fault name.
- Give the corrective checks in the SAME ORDER as the manual.
- Include important safety instructions from the manual.
- Include the source document and page information.
- Keep the answer concise but complete.
- If the manual does not contain the answer, say:
  "I cannot find this information in the available manual."

USER QUERY:
{query}

MANUAL EVIDENCE:
{result["text"]}

FINAL ANSWER:
"""


# ------------------------------------------------------------
# 3. Format prompt for Qwen3
# ------------------------------------------------------------

messages = [
    {
        "role": "user",
        "content": prompt
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)


# ------------------------------------------------------------
# 4. Tokenize and move to GPU
# ------------------------------------------------------------

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)


# ------------------------------------------------------------
# 5. Generate NORD answer
# ------------------------------------------------------------

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=280,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )


# ------------------------------------------------------------
# 6. Extract only newly generated tokens
# ------------------------------------------------------------

generated = output[0][inputs["input_ids"].shape[1]:]

answer = tokenizer.decode(
    generated,
    skip_special_tokens=True
).strip()


# ------------------------------------------------------------
# 7. Safety cleanup — remove accidental thinking blocks
# ------------------------------------------------------------

if "<think>" in answer:
    answer = answer.split("</think>")[-1].strip()


# ------------------------------------------------------------
# 8. Display final NORD response
# ------------------------------------------------------------

print("=" * 80)
print("USER QUERY")
print(query)
print("=" * 80)
print("NORD ANSWER")
print("=" * 80)
print(answer)
print("=" * 80)

In [ ]:
print("model:", model.device)
print("query:", query)
print("retrieved:", result["error_code"], "-", result["fault"])
print("evidence length:", len(result["text"]))

In [ ]:
# Simple NORD generation test

prompt = f"""You are NORD, an industrial troubleshooting assistant.

Use ONLY this manual evidence:

{result["text"]}

User question:
{query}

Give a concise final troubleshooting answer.
Do not show reasoning.
"""

messages = [{"role": "user", "content": prompt}]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

print("Generating...")

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

generated = output[0][inputs["input_ids"].shape[1]:]

answer = tokenizer.decode(
    generated,
    skip_special_tokens=True
).strip()

print("\n" + "=" * 80)
print("NORD ANSWER")
print("=" * 80)
print(answer)

In [ ]:
# ============================================================
# NORD — STRUCTURED MANUAL CONTEXT TEST
# ============================================================

nord_context = """
Fault Code: F001
Fault Name: DC Link Voltage Too High

Corrective Steps:
1. Measure mains at the main isolator terminals.
   Required: 400 V ±10% (360–440 V).
   If voltage is high, notify the facility electrician.

2. Power down and apply LOTO.
   Measure braking resistor at R+ and RB.
   Nominal resistance: 47 Ω ±10%.
   Replace if open or out of tolerance.
   Part No: DW-BRK-RES.

3. If the braking resistor is healthy:
   Increase BG.DECEL by 25%.
   Example: 0.2 s → 0.25 s.

4. If the fault persists:
   Request DeltaWorks service for a DC link capacitor capacitance test.

Source: test_manual_delta.pdf, page 4.
"""

prompt = f"""You are NORD, an industrial troubleshooting assistant.

Answer ONLY using the manual information below.

User question:
{query}

Manual information:
{nord_context}

Return a concise but complete answer.
Include all four corrective steps in order.
Do not show reasoning.
Do not use <think> tags.
Do not add outside information.
"""

messages = [{"role": "user", "content": prompt}]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=250,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

generated = output[0][inputs["input_ids"].shape[1]:]

answer = tokenizer.decode(
    generated,
    skip_special_tokens=True
).strip()

print("=" * 80)
print("NORD ANSWER")
print("=" * 80)
print(answer)